# Datencheck

Umsatzprognose, Gewinn/Verlust und Auslastung im Überblick - ähnlich den Grafiken der
monatlichen ZDF-Präsentation ("Zahlen, Daten, Fakten"), gespeist aus denselben
Datenquellen wie `01_dashboard.ipynb`. Dieses Notebook liest nur, es verändert nichts.

In [ ]:
# Nur in Google Colab: Projekt aus GitHub installieren, weil das lokale venv dort
# nicht zur Verfuegung steht. Lokal passiert hier nichts, dort liefert `uv sync` die
# Umgebung. Das Repository ist oeffentlich, deshalb braucht pip kein Token.
import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    # Fuer einen reproduzierbaren Lauf auf einen Tag setzen statt auf "main".
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"

    # Zwei Aufrufe.
    #
    # Der erste beschafft die Abhaengigkeiten, und nur die fehlenden: Colab pinnt
    # pandas 2.2.3 (google-colab) und numpy < 2.3 (numba); "pandas>=2.2" ist damit
    # erfuellt, pip laesst beide stehen. Fuer plotly gilt dasselbe - "plotly>=5" ist
    # von Colabs mitgelieferter Version erfuellt.
    #
    # Der zweite erneuert ausschliesslich unseren Code. --force-reinstall ist noetig,
    # weil die Versionsnummer ueber Commits hinweg 0.1.0 bleibt und pip die
    # Anforderung sonst fuer erfuellt haelt - "pip install git+...@main" laesst einen
    # installierten Stand dann unangetastet, ohne Fehlermeldung (verifiziert am
    # 24.08.2026, der Import schlug danach mit ModuleNotFoundError fehl).
    # --no-deps haelt pandas und numpy aus dem Reinstall heraus: ohne dieses Flag zog
    # der Aufruf pandas 3.0.5 und numpy 2.5.2 nach und brach google-colab 1.0.0 und
    # numba 0.61.2.
    #
    # Nach einem neuen Push zusaetzlich die Runtime neu starten. Sonst bleibt das alte
    # Paket im Speicher, und ein neuer Name in einem alten Modul endet als ImportError.
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
else:
    print("Lokale Installation")

## Parameter und Variablen für die Verarbeitung

In [ ]:
from datetime import date

stichtag = date.today()
abgeschlossene_monate = 12
horizont_monate = 3
gewinn_verlust_monate = 11
auslastung_monate = 12

## Daten laden

Dieselben Abrufe wie im Dashboard, dazu die Auslastung je Person und die Simulation für
den Prognosehorizont.

In [ ]:
from umsatzprognose import Dashboard

dashboard = Dashboard.laden(
    stichtag=stichtag,
    abgeschlossene_monate=abgeschlossene_monate,
    horizont_monate=horizont_monate,
    auslastung_monate=auslastung_monate,
)
dashboard.simuliere(monate=horizont_monate)

## Umsatzprognose

Bereits abgerechneter und gebuchter Umsatz, daran anschließend die Prognose für den
Horizont, zusammen mit Kosten und Ergebnis je Monat.

In [ ]:
dashboard.umsatzverlauf()

## Gewinn/Verlust je Monat

Umsatz minus Kosten für die letzten `gewinn_verlust_monate` abgeschlossenen Monate,
anschließend die Vorausschau über den mit `horizont_monate` konfigurierten
Prognosehorizont (gedämpfte Balken) - dieselbe Simulation wie in der Umsatzprognose.

In [ ]:
dashboard.gewinn_verlust_monatlich(monate=gewinn_verlust_monate)

## Kumulierter Gewinn/Verlust

Dieselben Monate, als aufsummierte Linie - zeigt, seit wann und wie deutlich ein
Gesamtminus (oder -plus) über den Zeitraum besteht. Die gestrichelte Fortsetzung ab dem
letzten Ist-Monat ist die Vorausschau über den Prognosehorizont.

In [ ]:
dashboard.gewinn_verlust_kumuliert(monate=gewinn_verlust_monate)

## Details je Monat

Dieselben Zahlen als Tabelle: Umsatz, Kosten und Gewinn je Monat, Historie und
Vorausschau über den Prognosehorizont.

In [ ]:
dashboard.umsatztabelle()

## Auslastung je Mitarbeiter

Anteil abrechenbarer Stunden (Clockodo-Billable-Status "abrechenbar" und "bereits
fakturiert") an der verfügbaren Kapazität, für den Stichtagsmonat - bereits beim Laden
für die letzten `auslastung_monate` Monate mit abgerufen.

In [ ]:
dashboard.auslastung_je_mitarbeiter(top=100)